# Phase 1 Diagnostic — Railgun Cooperation Failure (Colab)

Goal: prove the cooperation failure mode exists and is structural, before any algorithm work.

**Important difference from the plan in `colab_diagnostic_plan.md`:** RAILGUN does NOT use pogema. It uses its own `.map` files + `.mbin` data format, with LaCAM expert actions baked in per cell per timestep. So we don't need a separate Python LaCAM solver — the `.mbin` IS the expert reference.

Notebook structure:
- §0 Bootstrap: clone repo, install deps, build C++ tools + lacam3
- §1 Data prep: generate small map + .mbin dataset (smoke test)
- §2 Pretrained checkpoint: download from Google Drive
- §3 Sanity inference: load model, run on one sample
- §4 Per-cell diagnostic loop: log Fout, expert action, disagreement, criticality
- §5 Core analysis (Q1–Q5)
- §6 Findings

**Known unknowns** (flagged inline):
- Exact UNet arch params for the released checkpoint — try `first_layer_channels=64, blocks_per_stage=1` first.
- Whether C++ build works on Colab base image — should, but `apt install parallel` needs to succeed.
- Whether `gen_mapfile.sh` produces a 'warehouse'-like map or only mazes — repo seems maze-focused; we'll start there.

## §0 Bootstrap

In [ ]:
# 0.1 — System packages
!apt-get update -qq
!apt-get install -y -qq parallel cmake build-essential

In [ ]:
# 0.2 — Python deps (matches RAILGUN README + analysis tools)
!pip install -q torch numpy tqdm tensorboard psutil matplotlib pyyaml seaborn pandas pyarrow scikit-learn networkx gdown

In [ ]:
# 0.3 — Mount Drive (for persistent checkpoint + dataset storage)
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/railgun_followup'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/diagnostic_data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/figures', exist_ok=True)

In [ ]:
# 0.4 — Clone RAILGUN (Yimin's public repo)
import os
RAILGUN_DIR = '/content/railgun'

if not os.path.isdir(f'{RAILGUN_DIR}/models'):
    !git clone https://github.com/TachikakaMin/RAILGUN.git {RAILGUN_DIR}

assert os.path.isdir(f'{RAILGUN_DIR}/models'), (
    f'{RAILGUN_DIR}/models still not present after clone — check the cell output above.'
)
print(f'RAILGUN_DIR = {RAILGUN_DIR}')
!ls {RAILGUN_DIR} | head -20

In [ ]:
# 0.5 — Build lacam3 (C++ MAPF solver used for expert trajectories)
%cd {RAILGUN_DIR}/data_generation_LACAM
!if [ ! -d lacam3 ]; then git clone --recursive https://github.com/Kei18/lacam3.git; fi
%cd lacam3
!cmake -B build && cmake --build build -j 4
%cd {RAILGUN_DIR}

In [ ]:
# 0.6 — Build RAILGUN C++ extensions (feature construction, path conversion)
%cd {RAILGUN_DIR}/tools
!bash build.sh build
%cd {RAILGUN_DIR}

In [ ]:
# 0.7 — Verify imports
import sys
sys.path.insert(0, RAILGUN_DIR)

from models.unet import UNet
from MAPF_dataset_mbin import MAPFDataset as MAPFDatasetMbin
from tools.path_formation import path_formation
print('imports OK')

## §1 Data prep — generate a small dataset

Workflow: maps → distance maps → LaCAM-generated `.path` → `.mbin`.

In [ ]:
# 1.1 — Generate a small set of maps (override defaults to keep this fast)
# gen_mapfile.sh has tunables via env vars; if it's still slow, edit it down.
%cd {RAILGUN_DIR}
!bash gen_mapfile.sh
!ls data/map_files | head

In [ ]:
# 1.2 — Precompute distance maps (required by feature construction)
!python -m tools.precompute_distance_maps data/map_files
!ls data/distance_maps | head

In [ ]:
# 1.3 — Generate a small .mbin dataset for diagnostic
# 5 scenarios per (map, agent_count). Adjust if too slow / too sparse.
!python tools/generate_offline_data.py \
    --output_dir data/diagnostic_data \
    --num_scenarios 200 \
    --workers 4 \
    --time_limit 5 \
    --retry_limit 3 \
    --agent_counts 64 96 128 160 \
    --map_pattern "data/map_files/maze-*/*.map" \
    --scenarios_per_file 50

import glob
mbin_files = sorted(glob.glob('data/diagnostic_data/**/*.mbin', recursive=True))
print(f'Generated {len(mbin_files)} .mbin files')

## §2 Pretrained checkpoint

In [ ]:
# 2.1 — Download pretrained Railgun from Google Drive
# Link from RAILGUN/README.md
import gdown, os

ckpt_path = f'{PROJECT_DIR}/checkpoints/railgun_pretrained.pt'
if not os.path.exists(ckpt_path):
    gdown.download(
        'https://drive.google.com/uc?id=1r7tkbSBhterp3JK-yg6zsbDayfJketXe',
        ckpt_path,
        quiet=False,
    )
print(f'checkpoint at: {ckpt_path}')
print(f'size: {os.path.getsize(ckpt_path) / 1e6:.1f} MB')

## §3 Sanity inference

**Known unknown:** the released checkpoint's UNet width/depth. Defaults from `eval_test.py` are `first_layer_channels=64, blocks_per_stage=1`. If `load_state_dict` errors with size mismatch, try other combos (e.g., 32, 128, blocks 2).

In [ ]:
# 3.1 — Load model
import torch
from models.unet import UNet

device = 'cuda' if torch.cuda.is_available() else 'cpu'
FEATURE_DIM = 6
ACTION_DIM = 5

model = UNet(
    n_channels=FEATURE_DIM,
    n_classes=ACTION_DIM,
    first_layer_channels=64,   # adjust if checkpoint mismatch
    bilinear=False,
    blocks_per_stage=1,        # adjust if checkpoint mismatch
).to(device)

state = torch.load(ckpt_path, map_location=device)
if isinstance(state, dict) and 'model_state_dict' in state:
    state = state['model_state_dict']
model.load_state_dict(state)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'loaded UNet, {n_params/1e6:.2f}M params')

In [ ]:
# 3.2 — Load one sample, run forward pass
from MAPF_dataset_mbin import MAPFDataset as MAPFDatasetMbin

ds = MAPFDatasetMbin(mbin_files[:1], FEATURE_DIM, 'gradient', first_step=False)
sample = ds[0]
print({k: (v.shape if hasattr(v, 'shape') else v) for k, v in sample.items()})

feat = sample['feature'].unsqueeze(0).to(device)  # [1, C, H, W] expected by UNet
if feat.dim() == 4 and feat.shape[1] != FEATURE_DIM:
    # dataset returns [H, W, C] in some configs — permute if needed
    feat = feat.permute(0, 3, 1, 2).contiguous()

with torch.no_grad():
    logits = model(feat)  # [1, 5, H, W]
print('logits shape:', logits.shape)
print('expected: [1, 5, H, W]')

## §4 Per-cell diagnostic loop

For each agent-occupied cell, log:
- Railgun's full action distribution + argmax
- LaCAM expert action (from `.mbin`)
- Disagreement
- Criticality candidates: local agent density, graph betweenness (TODO), revisit count (TODO)

In [ ]:
# 4.1 — Helpers
import numpy as np
import torch.nn.functional as F

def local_density(mask_np, i, j, radius=3):
    H, W = mask_np.shape
    i0, i1 = max(0, i - radius), min(H, i + radius + 1)
    j0, j1 = max(0, j - radius), min(W, j + radius + 1)
    return mask_np[i0:i1, j0:j1].sum() - mask_np[i, j]  # exclude self

def per_cell_records(sample, logits, scenario_id, timestep):
    """Yield one dict per agent-occupied cell."""
    # logits: [1, 5, H, W]; sample['action']: [H, W] long; sample['mask']: [H, W] uint8
    probs = F.softmax(logits[0], dim=0).cpu().numpy()  # [5, H, W]
    argmax = probs.argmax(axis=0)
    expert = sample['action'].cpu().numpy()
    mask = sample['mask'].cpu().numpy()
    rows = []
    ii, jj = np.where(mask > 0)
    for i, j in zip(ii.tolist(), jj.tolist()):
        rows.append({
            'scenario_id': scenario_id,
            'timestep': timestep,
            'cell_i': i,
            'cell_j': j,
            'railgun_action_dist': probs[:, i, j].tolist(),
            'railgun_argmax': int(argmax[i, j]),
            'lacam_action': int(expert[i, j]),
            'disagreement': bool(argmax[i, j] != expert[i, j]),
            'local_agent_density': float(local_density(mask, i, j)),
        })
    return rows

In [ ]:
# 4.2 — Walk all scenarios in all .mbin files; collect per-cell records
from tqdm import tqdm
import pandas as pd

ds_full = MAPFDatasetMbin(mbin_files, FEATURE_DIM, 'gradient', first_step=False)
print(f'total steps: {len(ds_full)}')

records = []
with torch.no_grad():
    for idx in tqdm(range(len(ds_full))):
        s = ds_full[idx]
        feat = s['feature'].unsqueeze(0).to(device)
        if feat.shape[1] != FEATURE_DIM:
            feat = feat.permute(0, 3, 1, 2).contiguous()
        logits = model(feat)
        # scenario_id = file_name + '#' + scenario_idx — approximate via file_name + idx for now
        rows = per_cell_records(s, logits, scenario_id=s['file_name'], timestep=idx)
        records.extend(rows)

df = pd.DataFrame(records)
df.to_parquet(f'{PROJECT_DIR}/diagnostic_data/percell.parquet')
print(f'wrote {len(df):,} per-cell records')

In [ ]:
# 4.3 — Sanity
print(df.head())
print(f'\noverall disagreement rate: {df.disagreement.mean():.3f}')
print(f'\nrecords per cell-density bucket:')
print(df.groupby(pd.cut(df.local_agent_density, bins=[-1, 0, 2, 5, 10, 100])).disagreement.agg(['count', 'mean']))

## §5 Core analysis

In [ ]:
# 5.1 — Q1: Does disagreement correlate with local density?
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='disagreement', y='local_agent_density', ax=ax)
ax.set_title('Local agent density by disagreement')
plt.savefig(f'{PROJECT_DIR}/figures/density_vs_disagreement.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5.2 — Q2: Per-cell loss vs density
import torch.nn.functional as F

def per_cell_loss(dist, expert):
    return F.cross_entropy(
        torch.tensor([dist], dtype=torch.float32),
        torch.tensor([expert], dtype=torch.long),
    ).item()

df['per_cell_loss'] = df.apply(
    lambda r: per_cell_loss(r.railgun_action_dist, r.lacam_action), axis=1
)
print('Pearson(per_cell_loss, local_agent_density):',
      df[['per_cell_loss', 'local_agent_density']].corr().iloc[0, 1])

In [ ]:
# 5.3 — Q5: AUC of density predicting disagreement
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X = df[['local_agent_density']].values
y = df.disagreement.astype(int).values
lr = LogisticRegression().fit(X, y)
auc = roc_auc_score(y, lr.predict_proba(X)[:, 1])
print(f'AUC (density → disagreement): {auc:.4f}')
print('Target: > 0.65 for density to be a useful criticality proxy.')

In [ ]:
# 5.4 — Q4: Spatial heatmap of disagreement (per scenario_id)
# Pick one scenario_id (file_name) and render where disagreement clusters.
first_file = df.scenario_id.iloc[0]
sub = df[df.scenario_id == first_file]

Hmax = sub.cell_i.max() + 1
Wmax = sub.cell_j.max() + 1
heat = np.zeros((Hmax, Wmax))
cnt = np.zeros((Hmax, Wmax))
for _, r in sub.iterrows():
    heat[r.cell_i, r.cell_j] += float(r.disagreement)
    cnt[r.cell_i, r.cell_j] += 1
rate = heat / np.maximum(cnt, 1)

plt.figure(figsize=(8, 8))
plt.imshow(rate, cmap='hot')
plt.colorbar(label='disagreement rate')
plt.title(f'Disagreement heatmap\n{os.path.basename(first_file)}')
plt.savefig(f'{PROJECT_DIR}/figures/disagreement_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## §6 Findings

Fill in the values once §5 has run.

In [ ]:
from datetime import datetime

findings = f"""
# Phase 1 Diagnostic Findings — {datetime.now():%Y-%m-%d}

## Dataset
- .mbin files: {len(mbin_files)}
- per-cell records: {len(df):,}
- agent-occupied cells: {len(df):,} (per-cell loop only emits records for agent cells)

## Disagreement
- Overall rate: {df.disagreement.mean():.3f}
- AUC (density → disagreement): {auc:.4f}
- Pearson(per_cell_loss, density): {df[['per_cell_loss', 'local_agent_density']].corr().iloc[0,1]:.4f}

## Hypothesis check
- Disagreement concentrates in dense cells: [YES if AUC > 0.65 and density boxplot separates]
- Per-cell loss correlates with density: [WEAKLY if r < 0.1, supports loss-misalignment claim]
- Spatial clustering at junctions: [check heatmap visually]

## Decision
- If hypothesis supported: proceed to Phase 2 (negative result on decentralized shaping)
- If not: pivot framing
"""

with open(f'{PROJECT_DIR}/diagnostic_findings.md', 'w') as f:
    f.write(findings)
print(findings)

## What's still TODO (deferred for now)

Once §0–§5 run end-to-end with the released checkpoint, add:

- **`graph_betweenness`** criticality metric — precompute once per map using `networkx.betweenness_centrality` on the free-cell graph; join into `df` by `(map_name, cell_i, cell_j)`.
- **`lacam_revisit_count`** — within each scenario, count how many timesteps each cell appears in any agent's trajectory. Requires grouping records by scenario+cell.
- **Per-`agent_count` breakdown** — currently scenario_id == file_name. Parse `agent_num` out of the file path or extend the dataset to expose it, so we can plot disagreement vs n_agents (the headline scaling result).
- **Self-rollout vs teacher-forcing comparison** — current loop is teacher-forced (LaCAM-induced states). To compare to RAILGUN's self-rollout, use `tools.path_formation` (see `eval_test.py`) and log per-step disagreement against LaCAM's state at the same timestep.
- **Negative result (Cells 19–20 of original plan)** — Song et al. shaping fine-tune. Out of scope for Phase 1, kept in `colab_diagnostic_plan.md` for Phase 2.